# LULC - Train and predict

This notebook concatenates training data for many tiles for 2020, and trains a random forest model. It reads and writes to/from S3.

### Steps: 
1. Concat all tile training data CSVs from S3.
2. Filter outliers per class
3. Train model:
Try one model for all sites. (append all CSVs)
export model dump
in future we may need to make different models for different regions and year ranges.
train the model using the geomad of the year of the input products.

How many models do we need?
Pacific?
LS eras e.g. before 2013 and after.
Non-pacific model could be more diverse? might need more models.

In [ ]:
# Reload functions during development
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib inline

# Scientific core
import json
import math
import re
import subprocess
from datetime import date
from io import StringIO
from pathlib import Path

import boto3
import geopandas as gpd

# Machine learning
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Geospatial
from lonboard import BitmapTileLayer  # , Map, PolygonLayer, ScatterplotLayer
from lonboard.colormap import apply_categorical_cmap
from matplotlib.patches import Patch
from shapely import wkt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# from sklearn.metrics import f1_score, make_scorer
# from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from ldn.raster import FOCAL_FEATURES, INDEX_BANDS, MAD_BANDS, REFLECTANCE_BANDS
from ldn.training_data import NON_PACIFIC_TRAINING_TILES, PACIFIC_TRAINING_TILES, TEST_TILES, TRAINING_TILES
from ldn.typology import classes, classes_flipped, colors

# from ldn.aws import s3_client
from ldn.utils import (
    CLASS_ATTR,
    LULC_DATASET_ID,
    MODEL_VERSION,
    TEST_DATA_VERSION,
    TRAINING_DATA_VERSION,
    TRAINING_DATA_YEARS,
    WGS84,
    dataset_prefix,
    get_env_var,
    get_pathstyle_url_base,
    parse_tile_id,
)

BUCKET = get_env_var("BUCKET")
print(f"BUCKET: {BUCKET}")

print(f"Training tiles ({len(TRAINING_TILES)}): {TRAINING_TILES}")
print(f"Training data version: {TRAINING_DATA_VERSION}")
print(f"Model version: {MODEL_VERSION}")
print(f"Training data years: {TRAINING_DATA_YEARS}")

In [ ]:
# Gather all of the CSVs of training points for many AOIs.
# paths = f"training_data/{TRAINING_DATA_VERSION}/{tile_id}/{year}/samples_*.csv"
# files = glob.glob(paths)

# TODO: Could load from local dir

# aws_profile = os.environ.get("AWS_PROFILE")
aws_profile = "dep-staging-admin"  # TODO: Don't commit
# %env AWS_PROFILE
print(f"Using AWS profile: {aws_profile}")

aws_session = boto3.Session(profile_name=aws_profile)
print(f"Using AWS profile: {aws_session.profile_name}, region: {aws_session.region_name}")
s3_client = aws_session.client("s3", region_name=aws_session.region_name)

# List all training CSVs for this version
# The Makefile training and test data targets write both regions under the "dep" prefix
_owner = "dep"
_dataset_prefix = dataset_prefix(_owner, "ls", LULC_DATASET_ID)
resp = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=f"{_dataset_prefix}/training_data/{TRAINING_DATA_VERSION}/")
# This gets all regions (Pacific and non-Pacific). Can easily split by region here.
keys = [obj["Key"] for obj in resp.get("Contents", []) if obj["Key"].endswith("samples.csv")]
num_files_found = len(keys)
print(f"Found {num_files_found} CSV files")

# Filter to just the training tiles and years. Keys end with /{x}/{y}/{year}/samples.csv.
filtered_keys = []
for id, region, country_dict in TRAINING_TILES:
    tile_id_x, tile_id_y = parse_tile_id(id)
    for key in keys:
        # Zero-padding tile index values.
        if f"/{tile_id_x:03d}/{tile_id_y:03d}/" in key and key.split("/")[-2] in TRAINING_DATA_YEARS:
            filtered_keys.append(key)

num_files_found = len(filtered_keys)
print(f"Filtered to {num_files_found} CSV files for training tiles and years")
print(filtered_keys)

expected_num_files = len(TRAINING_TILES) * len(TRAINING_DATA_YEARS)
assert num_files_found == expected_num_files, (
    f"There should be {expected_num_files} training CSV files found in S3, not {num_files_found}."
)

# Stream each CSV from S3 into a DataFrame
dfs = {}
for key in filtered_keys:
    print(f"  Loading s3://{BUCKET}/{key}")
    body = s3_client.get_object(Bucket=BUCKET, Key=key)["Body"].read().decode("utf-8")
    region, tile_id_x, tile_id_y, sample_year = key.split("/")[-5:-1]
    # The region, tile and year are kept to describe the samples, but must not become features.
    dfs[(region, tile_id_x, tile_id_y, sample_year)] = pd.read_csv(StringIO(body)).assign(
        region=region, tile=f"{region}_{tile_id_x}_{tile_id_y}", year=int(sample_year)
    )
samples = pd.concat(dfs.values(), ignore_index=True)
print(f"Combined training data: {len(samples)} samples from {len(dfs)} files")

# Geometry only needed for viz, no per-region model yet
samples["geometry"] = samples["geometry"].apply(wkt.loads)
samples = gpd.GeoDataFrame(samples, geometry="geometry", crs=WGS84)

samples.drop(columns=["outlier"], inplace=True, errors="ignore")
# samples.drop(columns=["geometry"], inplace=True, errors="ignore")
samples.drop(columns=["time"], inplace=True, errors="ignore")
samples.drop(columns=["spatial_ref"], inplace=True, errors="ignore")
print(samples.columns)

print(samples.head())
# Samples per class for each region and year, as a markdown table.
counts = samples.groupby([CLASS_ATTR, "region", "year"]).size().unstack(["region", "year"], fill_value=0)
counts.loc["Total"] = counts.sum()
print(f"| Class | Name | {' | '.join(f'{region} {year}' for region, year in counts.columns)} |")
print(f"|---|---|{'---|' * len(counts.columns)}")
for c, row in counts.iterrows():
    print(f"| {c} | {classes_flipped.get(c, '')} | {' | '.join(f'{n:,}' for n in row)} |")

In [ ]:
def rgb_str_to_hex(rgb_str):
    r, g, b = map(int, re.findall(r"\d+", rgb_str))
    return "#{:02x}{:02x}{:02x}".format(r, g, b)


cmap = {cls: [int(v) for v in re.findall(r"\d+", rgb)] for cls, rgb in colors.items()}
satellite = BitmapTileLayer(
    data="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    tile_size=256,
    max_zoom=19,
)
regions_cmap = {
    "pacific": [0, 208, 255],
    "non-pacific": [55, 255, 20],
}
sids_tiles = gpd.read_file("../ldn/sids_all_tiles.geojson")
training_tile_ids = {parse_tile_id(t[0]) for t in TRAINING_TILES}
is_training = [xy in training_tile_ids for xy in zip(sids_tiles["x_index"], sids_tiles["y_index"])]
tile_colors = apply_categorical_cmap(sids_tiles["region"], regions_cmap)
tile_colors = np.column_stack([tile_colors, np.where(is_training, 255, 60).astype(np.uint8)])
# tiles = PolygonLayer.from_geopandas(
#     sids_tiles,
#     filled=False,
#     get_line_color=tile_colors,
#     line_width_min_pixels=1,
# )
# points = ScatterplotLayer.from_geopandas(
#     samples,
#     get_fill_color=apply_categorical_cmap(samples[CLASS_ATTR], cmap),
#     radius_min_pixels=3,
# )
# Map([satellite, tiles, points])

### Visualise training samples per class per tile and year

In [ ]:
# One panel per training tile, with a bar per class for each year. Tiles that lack a class in one year show a gap.
hex_colors = {cls: rgb_str_to_hex(rgb) for cls, rgb in colors.items()}
class_ids = sorted(samples[CLASS_ATTR].unique())
bar_width = 0.8 / len(TRAINING_DATA_YEARS)
year_alphas = np.linspace(1, 0.35, len(TRAINING_DATA_YEARS))
y_max = max(df[CLASS_ATTR].value_counts().max() for df in dfs.values()) * 1.05

ncols = 5
regions = {"Pacific": PACIFIC_TRAINING_TILES, "Non-Pacific": NON_PACIFIC_TRAINING_TILES}
nrows = {name: math.ceil(len(tiles) / ncols) for name, tiles in regions.items()}
fig = plt.figure(figsize=(4 * ncols, 2.5 * sum(nrows.values()) + 1), layout="constrained")

for subfig, (name, tiles) in zip(fig.subfigures(len(regions), 1, height_ratios=list(nrows.values())), regions.items()):
    subfig.suptitle(name, fontsize=16, fontweight="bold")
    axes = subfig.subplots(nrows[name], ncols, squeeze=False)
    for ax, (tile_id, region, country) in zip(axes.flat, tiles):
        for i, (year, alpha) in enumerate(zip(TRAINING_DATA_YEARS, year_alphas)):
            counts = (
                dfs[(region, *tile_id.split("_"), year)][CLASS_ATTR].value_counts().reindex(class_ids, fill_value=0)
            )
            ax.bar(
                np.arange(len(class_ids)) + (i - (len(TRAINING_DATA_YEARS) - 1) / 2) * bar_width,
                counts,
                bar_width,
                color=[hex_colors[c] for c in class_ids],
                alpha=alpha,
                edgecolor="black",
                linewidth=0.5,
            )
        ax.set_title(f"{next(iter(country))} {tile_id}", fontsize=10)
        ax.set_xticks([])
        ax.set_ylim(0, y_max)
    for ax in axes.flat[len(tiles) :]:
        ax.set_visible(False)
    for ax in axes[:, 0]:
        ax.set_ylabel("samples")

class_handles = [Patch(color=hex_colors[c], label=classes_flipped[c]) for c in class_ids]
year_handles = [Patch(color="grey", alpha=a, label=y) for y, a in zip(TRAINING_DATA_YEARS, year_alphas)]
fig.legend(
    handles=class_handles + year_handles, loc="outside lower center", ncol=len(class_handles) + len(year_handles)
)
fig.suptitle(f"Training samples per class, training data {TRAINING_DATA_VERSION}", fontsize=18)

# Saved with the version's test reports, so versions can be compared.
chart_path = Path(f"LULC_Testing/reports_test/{TRAINING_DATA_VERSION}/training_samples_per_class.png")
chart_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(chart_path, bbox_inches="tight")
plt.show()

In [ ]:
# from shapely.geometry import box

# # Define regions to split models by.
# max_y = 90
# min_y = -90

# split_x_1 = -180
# split_x_2 = -107
# split_x_3 = -40
# # split_x_4 = 90 # Puts Singapore and Timor-Leste in Pacific2
# split_x_4 = 128  # Puts Singapore and Timor-Leste in Africa
# split_x_5 = 180

# regions = {
#     # West to East.
#     "pacific1": box(split_x_1, min_y, split_x_2, max_y),
#     "caribbean": box(split_x_2, min_y, split_x_3, max_y),
#     "africa": box(split_x_3, min_y, split_x_4, max_y),
#     "pacific2": box(split_x_4, min_y, split_x_5, max_y),
# }

# # Build a GeoDataFrame of region boxes and spatial-join to samples
# regions_gdf = gpd.GeoDataFrame(
#     {"region": regions.keys()},
#     geometry=list(regions.values()),
#     crs=WGS84,
# )
# samples = gpd.sjoin(
#     samples, regions_gdf[["region", "geometry"]], how="left", predicate="intersects"
# )
# samples.drop(columns=["index_right"], inplace=True, errors="ignore")

# print("Samples per region:")
# print(samples["region"].value_counts())
# print(f"Samples without a region: {samples['region'].isna().sum()}")

# # TODO: Use this region logic once we split the model per region.
# # TODO: Count classes per region. Many will be missing.

In [ ]:
# # Make different models for time periods.
# years = [year for year in range(2000, 2025)]  # 2000 to 2024
# print(years)

# # TODO: Verify these splits.
# # In our GeoMAD logic we use <= LS7_YEAR_THRESHOLD as a condition to use a buffered year and T1 and T2 data.
# time_periods = {
#     "tm_etm": [year for year in range(2000, 2003)],  # L5 + L7 pre-SLC failure
#     "slc_off": [year for year in range(2003, 2013)],  # L7 SLC-off, L5 aging/gone
#     "oli": [year for year in range(2013, 2025)],  # L8/L9, modern radiometry
# }
# print(time_periods)

In [ ]:
# # TODO: Cross regions and periods
# region_periods = {}

## Again filter for outliers (for the combined data for many tiles).

In [ ]:
# Anomaly detection. 3 standard deviations from the mean is a good guideline for identifying outliers (in continuous
# data). This may not apply because all pixels are 1 class (categorical data).

# # TODO: here balance concatenated training data by class e.g.
# if one class has a huge amount of samples (after contatenation).
# # TODO: in future think about region-splitting per model.
# # TODO: Do a sample distribution check per class AND country/region.
# # This can decide which countries should be included to train the model/model per region.
# # Focus on getting all classes e.g. other.

# TODO: Use ldn/training_data.py's filter_outliers function.
# from ldn.training_data import filter_outliers
# TODO: Run a baseline version and then test if outlier filtering improves the model and classification.

### Test correlation between features. Exclude >95% correlated features.

In [ ]:
# exclude_cols = ['lulc', 'geometry']
# feature_cols = [c for c in samples.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(samples[c])]

# corr = samples[feature_cols].corr().abs()

# # Upper triangle mask (ignore self-correlation on the diagonal)
# upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))

# # Find pairs above 0.95
# high_corr_pairs = [
#     (col, row, upper.loc[row, col])
#     for col in upper.columns
#     for row in upper.index
#     if upper.loc[row, col] > 0.95
# ]

# if high_corr_pairs:
#     print("Highly correlated feature pairs (>0.95):")
#     for a, b, r in sorted(high_corr_pairs, key=lambda x: -x[2]):
#         print(f"  {a:12s} ↔ {b:12s}  r={r:.3f}")

#     # Drop the second feature in each pair (keep first encountered)
#     to_drop = {b for _, b, _ in high_corr_pairs}
#     print(f"\nDropping {len(to_drop)} redundant feature(s): {to_drop}")
#     samples.drop(columns=to_drop, inplace=True, errors='ignore')
# else:
#     print("No feature pairs with correlation > 0.95 — keeping all features.")

# # Heatmap using matplotlib only
# fig, ax = plt.subplots(figsize=(12, 10))
# im = ax.imshow(corr.values, cmap="coolwarm", vmin=0, vmax=1)
# ax.set_xticks(range(len(feature_cols)))
# ax.set_yticks(range(len(feature_cols)))
# ax.set_xticklabels(feature_cols, rotation=45, ha="right", fontsize=7)
# ax.set_yticklabels(feature_cols, fontsize=7)
# for i in range(len(feature_cols)):
#     for j in range(len(feature_cols)):
#         ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=5,
#                 color="white" if corr.values[i, j] > 0.7 else "black")
# plt.colorbar(im, ax=ax, label="Absolute correlation")
# ax.set_title("Feature Correlation (absolute)")
# plt.tight_layout()
# plt.show()

## Train the model

In [ ]:
# Define features and target
# The year describes the samples (2017-2021) and must not become a feature.
# Training data 0-2-1 also has the features that 0-2-2 removed (issue #164), so they are dropped to reuse it.
removed_features = ["northness", "eastness", "elevation_tpi_51"]
removed_features += [c for c in samples.columns if c.split("_")[0] in ("ndti", "bsi")]
feature_cols = [
    c
    for c in samples.columns
    if c not in {CLASS_ATTR, "region", "year", *removed_features} and pd.api.types.is_numeric_dtype(samples[c])
]
# The model must have exactly the features that prediction makes (make_features in ldn/raster.py).
expected_features = REFLECTANCE_BANDS + MAD_BANDS + INDEX_BANDS + ["elevation", "slope"] + FOCAL_FEATURES
assert set(feature_cols) == set(expected_features), set(feature_cols) ^ set(expected_features)
print(f"Removed {len(removed_features)} features, training on {len(feature_cols)}")

# Pass DataFrames (not .values) so scikit-learn records feature names.
# This avoids the "fitted without feature names" warning at prediction time.
X_train = samples[feature_cols]
y_train = samples[CLASS_ATTR]

# print(f"Classes: {np.unique(y_train)}")

# The grid is off for production. Its first runs chose min_samples_leaf=10 (see the README).
# # Hyperparameter grid, scored by holding out whole tiles (both years together), like predicting an unseen tile.
# # This replaces OOB, which is optimistic because it scores points with trees trained on their spatial neighbours.
# # Stratified, so every fold has the hard classes, which are in few tiles. Training tiles only: the test tiles stay for
# # the final score. min_samples_leaf > 1 stops trees memorising noisy agreement labels and near-duplicate neighbours.
# hard_classes = [classes[c] for c in ("Cropland", "Wetland", "Other")]
# grid = GridSearchCV(
#     RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1),
#     # 10 was best of 1, 3, 5 and 10, and class_weight="balanced_subsample" made no difference.
#     {"min_samples_leaf": [10, 20]},
#     scoring={
#         "macro_f1": "f1_macro",
#         "hard_f1": make_scorer(f1_score, labels=hard_classes, average="macro", zero_division=0),
#     },
#     cv=StratifiedGroupKFold(5, shuffle=True, random_state=0),
#     refit=False,
# ).fit(X_train, y_train, groups=samples["tile"])
# cv = pd.DataFrame(grid.cv_results_)
# cv_table = cv[["params", "mean_test_macro_f1", "std_test_macro_f1", "mean_test_hard_f1", "std_test_hard_f1"]]
# print(cv_table.to_string())

# # Fold spread across tiles is large, so take the largest min_samples_leaf within one std of the best hard-class F1.
# best = cv.loc[cv["mean_test_hard_f1"].idxmax()]
# within = cv[cv["mean_test_hard_f1"] >= best["mean_test_hard_f1"] - best["std_test_hard_f1"]]
# best_params = within.sort_values(["param_min_samples_leaf", "mean_test_hard_f1"], ascending=False).iloc[0]["params"]
# print(f"Chosen parameters: {best_params}")

# 500 estimators could be needed for a many country model.
# n_jobs=-1 builds and predicts trees on all cores. Saved in the model, so prediction uses the pod's CPU limit.
model = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    # n_estimators=500, class_weight="balanced", random_state=42, n_jobs=-1, **best_params
).fit(X_train, y_train)

# Feature importance
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Feature importances:")
print(importances)

In [ ]:
# RF Classification Report - check the confidence map - how many trees voted the same way. 50% confidence is standard.
# Post-processing can be done e.g. we smooth these temporal flips to one class over time.
# Post-processing should be minimal.
# Another post-processing method is to smooth any single pixel surrounded in space or time by another
# class to that same class.

In [ ]:
# One model per version, trained on every year in TRAINING_DATA_YEARS.
model_path = f"models/{MODEL_VERSION}"
s3_key = f"{model_path}/lulc_random_forest_model.joblib"
feature_importances_s3_key = f"{model_path}/feature_importances.csv"

joblib_path = Path(f"../ldn/{s3_key}")
feature_importances_path = Path(f"../ldn/{feature_importances_s3_key}")
joblib_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, joblib_path)

importances.to_csv(feature_importances_path)

# Write the model to S3 so it can be loaded in prediction step.
s3_key = f"{_dataset_prefix}/{s3_key}"
feature_importances_s3_key = f"{_dataset_prefix}/{feature_importances_s3_key}"
# Takes a while because the model is ~650 MB
s3_client.upload_file(str(joblib_path), BUCKET, s3_key)
s3_client.upload_file(str(feature_importances_path), BUCKET, feature_importances_s3_key)

s3_url = f"{get_pathstyle_url_base(BUCKET)}/{s3_key}"
print(f"Uploaded model to {s3_url}")
feature_importances_s3_url = f"{get_pathstyle_url_base(BUCKET)}/{feature_importances_s3_key}"
print(f"Uploaded feature importances to {feature_importances_s3_url}")

# # Load the model
# model = joblib.load(joblib_path)

In [ ]:
# Check the model on the held-out test points before running LULC.
# LULC_Testing/LULC_Test.ipynb scores the final product output.
# Score the model on the frozen held-out test tiles (never used for training)
resp = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=f"{_dataset_prefix}/test_data/{TEST_DATA_VERSION}/")
test_keys = [obj["Key"] for obj in resp.get("Contents", []) if obj["Key"].endswith("samples.csv")]
print(f"Found test data for {len(test_keys)} of {len(TEST_TILES)} test tiles")
assert test_keys, f"No test data found for TEST_DATA_VERSION {TEST_DATA_VERSION}."

test_df = pd.concat(
    [
        pd.read_csv(StringIO(s3_client.get_object(Bucket=BUCKET, Key=k)["Body"].read().decode())).assign(
            region=k.split("/")[-5]
        )
        for k in test_keys
    ]
)
print(test_df[CLASS_ATTR].value_counts().sort_index())

y_test_held_out = test_df[CLASS_ATTR]
y_pred_held_out = model.predict(test_df[feature_cols])
present = np.unique(np.concatenate([y_test_held_out, y_pred_held_out]))
target_names = [k for k, v in sorted(classes.items(), key=lambda x: x[1]) if v in present]
print(classification_report(y_test_held_out, y_pred_held_out, target_names=target_names, labels=present))

# Rows are the true class, columns the predicted class.
cm = confusion_matrix(y_test_held_out, y_pred_held_out, labels=present)
print(pd.DataFrame(cm, index=target_names, columns=target_names))

# Count and share of each reference class, as in LULC_Testing/LULC_Test.ipynb. The diagonal share is recall.
test_years = ", ".join(sorted({k.split("/")[-2] for k in test_keys}))
row_share = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(row_share, cmap="Blues", vmin=0, vmax=1)
for i in range(len(present)):
    for j in range(len(present)):
        colour = "white" if row_share[i, j] > 0.5 else "black"
        ax.text(j, i, f"{cm[i, j]}\n{row_share[i, j]:.0%}", ha="center", va="center", fontsize=8, color=colour)
ax.set_xticks(range(len(present)), target_names, rotation=45, ha="right")
ax.set_yticks(range(len(present)), target_names)
ax.set_xlabel(f"Model {MODEL_VERSION} ({test_years})")
ax.set_ylabel(f"Reference, test data {TEST_DATA_VERSION} ({test_years})")
ax.set_title("Confusion matrix (count and share of reference class)")
fig.tight_layout()
cm_path = joblib_path.parent / "confusion_matrix.png"
fig.savefig(cm_path, dpi=150)
plt.show()

# Metrics are stored next to the model (locally and in S3), so model versions can be compared by code.
metrics = {
    "model_version": MODEL_VERSION,
    "created": date.today().isoformat(),
    "git_commit": subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
    "training_data_version": TRAINING_DATA_VERSION,
    "training_data_years": TRAINING_DATA_YEARS,
    "test_data_version": TEST_DATA_VERSION,
    "random_forest_params": model.get_params(),
    "n_features": len(feature_cols),
    # "cv": {"chosen_params": best_params, "results": cv_table.to_dict("records")},
    "training_samples_per_class": {classes_flipped[c]: n for c, n in y_train.value_counts().sort_index().items()},
    "test_samples_per_class": {classes_flipped[c]: n for c, n in y_test_held_out.value_counts().sort_index().items()},
    "test_report": classification_report(
        y_test_held_out, y_pred_held_out, target_names=target_names, labels=present, output_dict=True
    ),
    # Next to metrics.json, locally and in S3.
    "test_confusion_matrix": cm_path.name,
    # Per region, so a weak region (e.g. Pacific cropland) isn't hidden by the combined score.
    "test_report_per_region": {
        region: classification_report(
            df[CLASS_ATTR],
            y_pred_held_out[test_df["region"] == region],
            labels=present,
            target_names=target_names,
            output_dict=True,
            zero_division=0,
        )
        for region, df in test_df.groupby("region")
    },
}
metrics_path = joblib_path.parent / "metrics.json"
# NumPy numbers aren't JSON serialisable, so convert them to Python numbers.
metrics_path.write_text(json.dumps(metrics, indent=2, default=lambda x: x.item()))
print(f"Wrote metrics to {metrics_path}")
metrics_s3_key = f"{_dataset_prefix}/{model_path}/metrics.json"
s3_client.upload_file(str(metrics_path), BUCKET, metrics_s3_key)
s3_client.upload_file(str(cm_path), BUCKET, f"{_dataset_prefix}/{model_path}/{cm_path.name}")
print(f"Uploaded metrics to {get_pathstyle_url_base(BUCKET)}/{metrics_s3_key}")

# TODO: Test against other years too.

If classes are confused e.g. crop and grassland, then increase the strictness of agreement needed for that class.

Will the model be released? If yes, Wei Ji recommends another Treelite instead of joblib. It is more library-agnostic.https://treelite.readthedocs.io/en/latest/index.html. It's actually not a file format, but an interchange format, so you could load a model from scikit-learn and then use it in a different library (e.g. nvForest). Something to think about if you'll be releasing the trained model, but ok to ignore if not.